### 1) Colab 작업 폴더 준비
드라이브의 프로젝트를 Colab 로컬(`/content`)로 복사하고 `data.zip`을 풀어서 실행 가능한 상태를 만듭니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DRIVE = "/content/drive/MyDrive/Intelligent Multimedia Systems Team Project/downstream_robustness_prediction"
DATA_ZIP_DRIVE = "/content/drive/MyDrive/Intelligent Multimedia Systems Team Project/downstream_robustness_prediction/data.zip"

!rm -rf /content/downstream_robustness_prediction
!rsync -av --exclude 'data.zip' "$PROJECT_DRIVE/" /content/downstream_robustness_prediction/
!unzip -q "$DATA_ZIP_DRIVE" -d /content/downstream_robustness_prediction/

%cd /content/downstream_robustness_prediction

Mounted at /content/drive
sending incremental file list
created directory /content/downstream_robustness_prediction
./
STRUCTURE.md
configs/
configs/config.yaml
data/
data/interim/
data/interim/clips/
data/interim/clips/obfuscated/
data/interim/clips/obfuscated/blur/
data/interim/clips/obfuscated/h264_local/
data/interim/clips/obfuscated/pixelate/
data/interim/clips/original/
data/interim/logs/
data/interim/manifests/
data/interim/manifests/clip_manifest.csv
data/processed/
data/processed/evaluation/
data/processed/evaluation/metrics/
data/processed/evaluation/yolo_deepsort/
data/processed/model_inputs/
data/processed/model_inputs/fusion/
data/processed/model_inputs/param_only/
data/processed/model_inputs/visual_only/
data/processed/targets/
data/raw/
data/raw/MOT17/
data/raw/MOT17/dataset_split.txt
experiments/
experiments/baseline_param/
experiments/baseline_visual/
experiments/fusion_multitask/
scripts/
scripts/01_generate_original_clips.py
scripts/02_apply_obfuscation.py
scripts/03

### 2) 필수 패키지 설치 + 스크립트 문법 체크
실행에 필요한 라이브러리를 설치하고, `03_generate_targets.py`가 문법적으로 정상인지 확인합니다.

In [2]:
!pip install -q ultralytics deep-sort-realtime lap opencv-python pandas numpy pyyaml tqdm scipy
!python -m py_compile scripts/03_generate_targets.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 86.0 MB/s eta 0:00:00


### 3) 실행 전 입력 데이터 최소 점검
`clip_manifest.csv`와 `config.yaml` 존재 여부를 먼저 확인합니다.

In [3]:
from pathlib import Path
import pandas as pd

root = Path('/content/downstream_robustness_prediction')
m = root / 'data/interim/manifests/clip_manifest.csv'

print("manifest exists:", m.exists())
if m.exists():
    df = pd.read_csv(m)
    print("rows:", len(df))
    print("degradation_type counts:", df['degradation_type'].value_counts(dropna=False).to_dict())

print("config exists:", (root / 'configs/config.yaml').exists())

manifest exists: True
rows: 3872
degradation_type counts: {'blur': 1730, 'pixelate': 1384, 'h264_local': 412, 'original': 346}
config exists: True


### 4) TrackEval NumPy 호환 패치 (필수)
Colab(Python 3.12 + NumPy 2.x)에서 TrackEval의 `np.float/np.int/np.bool` 호환 문제를 막습니다.

In [4]:
from pathlib import Path
import subprocess

root = Path('/content/downstream_robustness_prediction')
trackeval_dir = root / 'TrackEval'
run_script = trackeval_dir / 'scripts' / 'run_mot_challenge.py'

if not run_script.exists():
    print('TrackEval not found. Cloning once before patch...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/JonathonLuiten/TrackEval.git', str(trackeval_dir)],
        check=True
    )

txt = run_script.read_text(encoding='utf-8')
needle = 'import trackeval  # noqa: E402'
shim = """import numpy as np  # noqa: E402
if not hasattr(np, "float"):
    np.float = float  # type: ignore[attr-defined]
if not hasattr(np, "int"):
    np.int = int  # type: ignore[attr-defined]
if not hasattr(np, "bool"):
    np.bool = bool  # type: ignore[attr-defined]
"""

if 'np.float = float' not in txt:
    if needle not in txt:
        raise RuntimeError(f'Expected marker not found in {run_script}')
    txt = txt.replace(needle, shim + '\n' + needle)
    run_script.write_text(txt, encoding='utf-8')

txt2 = run_script.read_text(encoding='utf-8')
print('np.float shim:', 'np.float = float' in txt2)
print('np.int shim:', 'np.int = int' in txt2)
print('np.bool shim:', 'np.bool = bool' in txt2)

TrackEval not found. Cloning once before patch...
np.float shim: True
np.int shim: True
np.bool shim: True


### 5) Smoke 테스트 실행 (`smoke_fix2`)
전체 돌리기 전에 작은 범위(원본 2개 + obf 10개)로 성공 여부를 빠르게 확인합니다.

In [5]:
%cd /content/downstream_robustness_prediction
!python scripts/03_generate_targets.py \
  --semantic_run_id smoke_fix2 \
  --max_original_clips 2 \
  --max_obf_clips 10 \
  --save_every 1 \
  --yolo_device 0 \
  --strict_drive_mount true \
  --drive_sync_dir "/content/drive/MyDrive/Intelligent Multimedia Systems Team Project/downstream_robustness_prediction/data/interim/manifests"

/content/downstream_robustness_prediction
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] Project root: /content/downstream_robustness_prediction
[INFO] Drive sync dir: /content/drive/MyDrive/Intelligent Multimedia Systems Team Project/downstream_robustness_prediction/data/interim/manifests
[INFO] Loading detector: yolov8n.pt
[INFO] Step A: Computing/caching P_orig for original clips.
Original: 100% 2/2 [00:37<00:00, 18.79s/it]
[INFO] Step B: Computing P_anon and deltas for obfuscated clips.
Obfuscated: 100% 10/10 [00:52<00:00,  5.27s/it]
[INFO] Final save/sync.
[INFO] Canonical run_config_snapshot.json promoted after successful completion.
[INFO] Canonical target_stats.csv promoted after successful completion.
[SUCCESS]

### 6) Smoke 결과 확인
`original_eval` 실패가 사라졌는지, cache/target 산출물이 실제로 생겼는지 확인합니다.

In [6]:
from pathlib import Path
import pandas as pd

root = Path('/content/downstream_robustness_prediction')
manifests = root / 'data/interim/manifests'
logs = root / 'data/interim/logs'

target_path = manifests / 'target_manifest_smoke_fix2.csv'
cache_path = manifests / 'original_performance_cache_smoke_fix2.csv'
stats_path = manifests / 'target_stats_smoke_fix2.csv'
fail_path = logs / 'target_failures_smoke_fix2.csv'

for p in [target_path, cache_path, stats_path, fail_path]:
    print(p, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

if target_path.exists():
    print('target rows:', len(pd.read_csv(target_path)))
if cache_path.exists():
    print('cache rows:', len(pd.read_csv(cache_path)))
if fail_path.exists():
    f = pd.read_csv(fail_path)
    print('failure stage counts:', f['stage'].value_counts(dropna=False).to_dict())
    print('original_eval count:', int((f['stage'] == 'original_eval').sum()))

/content/downstream_robustness_prediction/data/interim/manifests/target_manifest_smoke_fix2.csv exists= True size= 1818
/content/downstream_robustness_prediction/data/interim/manifests/original_performance_cache_smoke_fix2.csv exists= True size= 169
/content/downstream_robustness_prediction/data/interim/manifests/target_stats_smoke_fix2.csv exists= True size= 307
/content/downstream_robustness_prediction/data/interim/logs/target_failures_smoke_fix2.csv exists= True size= 101
target rows: 10
cache rows: 2
failure stage counts: {}
original_eval count: 0


### 7) Canonical full-run 전 이전 결과 정리
과거 실패 산출물이 남아 결과 해석을 방해하지 않도록 canonical 파일만 지웁니다.

In [7]:
from pathlib import Path

root = Path('/content/downstream_robustness_prediction')
cleanup_targets = [
    root / 'data/interim/manifests/target_manifest.csv',
    root / 'data/interim/manifests/original_performance_cache.csv',
    root / 'data/interim/manifests/target_stats.csv',
    root / 'data/interim/manifests/target_stats.partial.csv',
    root / 'data/interim/manifests/run_config_snapshot.json',
    root / 'data/interim/manifests/run_config_snapshot.partial.json',
    root / 'data/interim/logs/target_failures.csv',
]

for p in cleanup_targets:
    if p.exists():
        p.unlink()
        print('removed:', p)
    else:
        print('skip (not found):', p)

skip (not found): /content/downstream_robustness_prediction/data/interim/manifests/target_manifest.csv
skip (not found): /content/downstream_robustness_prediction/data/interim/manifests/original_performance_cache.csv
skip (not found): /content/downstream_robustness_prediction/data/interim/manifests/target_stats.csv
skip (not found): /content/downstream_robustness_prediction/data/interim/manifests/target_stats.partial.csv
skip (not found): /content/downstream_robustness_prediction/data/interim/manifests/run_config_snapshot.json
skip (not found): /content/downstream_robustness_prediction/data/interim/manifests/run_config_snapshot.partial.json
skip (not found): /content/downstream_robustness_prediction/data/interim/logs/target_failures.csv


### 8) Canonical full-run 실행
최종 결과 파일(`target_manifest.csv` 등)을 생성합니다.

In [8]:
%cd /content/downstream_robustness_prediction
!python scripts/03_generate_targets.py \
  --save_every 20 \
  --yolo_device 0 \
  --strict_drive_mount true \
  --drive_sync_dir "/content/drive/MyDrive/Intelligent Multimedia Systems Team Project/downstream_robustness_prediction/data/interim/manifests"

/content/downstream_robustness_prediction
[INFO] Project root: /content/downstream_robustness_prediction
[INFO] Drive sync dir: /content/drive/MyDrive/Intelligent Multimedia Systems Team Project/downstream_robustness_prediction/data/interim/manifests
[INFO] Loading detector: yolov8n.pt
[INFO] Step A: Computing/caching P_orig for original clips.
Original: 100% 346/346 [21:28<00:00,  3.72s/it]
[INFO] Step B: Computing P_anon and deltas for obfuscated clips.
Obfuscated: 100% 3526/3526 [3:14:34<00:00,  3.31s/it]
[INFO] Final save/sync.
[INFO] Canonical run_config_snapshot.json promoted after successful completion.
[INFO] Canonical target_stats.csv promoted after successful completion.
[SUCCESS] Target rows: 3526
[SUCCESS] Baseline cache rows: 346
[INFO] Failure rows: 0
[INFO] Skip stats: {'original_low_active_trajectories': 0, 'obfuscated_low_active_trajectories': 0, 'obfuscated_existing_target': 0, 'obfuscated_missing_baseline': 0, 'obfuscated_low_baseline_filtered': 0, 'contract_violatio

### 9) Canonical 결과 확인
최종 산출물이 비어 있지 않은지, 실패 스테이지 분포가 정상인지 확인합니다.

In [9]:
from pathlib import Path
import pandas as pd

root = Path('/content/downstream_robustness_prediction')
manifests = root / 'data/interim/manifests'
logs = root / 'data/interim/logs'

target_path = manifests / 'target_manifest.csv'
cache_path = manifests / 'original_performance_cache.csv'
stats_path = manifests / 'target_stats.csv'
fail_path = logs / 'target_failures.csv'

for p in [target_path, cache_path, stats_path, fail_path]:
    print(p, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

if target_path.exists():
    t = pd.read_csv(target_path)
    print('target rows:', len(t))
if cache_path.exists():
    c = pd.read_csv(cache_path)
    print('cache rows:', len(c))
if stats_path.exists():
    s = pd.read_csv(stats_path)
    g = s[s['split'] == 'GLOBAL']
    if not g.empty:
        print('GLOBAL included_targets:', int(g['included_targets'].iloc[0]))
        print('GLOBAL excluded_eval_failure:', int(g['excluded_eval_failure'].iloc[0]))
if fail_path.exists():
    f = pd.read_csv(fail_path)
    print('failure stage counts:', f['stage'].value_counts(dropna=False).to_dict())
    print('obfuscated_missing_baseline_from_orig_eval_failure:',
          int((f['stage'] == 'obfuscated_missing_baseline_from_orig_eval_failure').sum()))

/content/downstream_robustness_prediction/data/interim/manifests/target_manifest.csv exists= True size= 599191
/content/downstream_robustness_prediction/data/interim/manifests/original_performance_cache.csv exists= True size= 21349
/content/downstream_robustness_prediction/data/interim/manifests/target_stats.csv exists= True size= 456
/content/downstream_robustness_prediction/data/interim/logs/target_failures.csv exists= True size= 101
target rows: 3526
cache rows: 346
GLOBAL included_targets: 3526
GLOBAL excluded_eval_failure: 0
failure stage counts: {}
obfuscated_missing_baseline_from_orig_eval_failure: 0
